In [1]:
from IPython.display import HTML
with open("custom/style.css", "r") as f:
    custom_style = f.read()
hide_cell_style = "<style>.jp-Cell:has(.hide-me) { display: yes; }</style>"
HTML(f"<div class='hide-me'></div><style>{custom_style}</style>{hide_cell_style}")

<h1><center><font color='blue'>Introduction to Parallel Programming with MPI<br><br>MPI Point-to-Point Communication: Nonblocking</font></center></h1>

# <font color='blue'>MPI Point-to-Point Communication: Nonblocking</font>

## Crystalline and Molecular Structures

<img src="figs/energy-landscape.svg" alt="energy-landscape.svg" style="float: right; margin-right: 15px;" width="45%">
<!--```{image} figs/energy-landscape.svg
:width: 600px
```-->

- Challenges in structure search: <font color='green'>chemistry</font> and <font color='green'>material science</font>
  - Local minimization
    - Computational cost: <font color='green'>variable</font>
  - Many local minima
    - <font color='red'>Exponential increase</font> with respect to system size
  - Global optimization methods
    - Deterministic and stochastic walker
- Parallel computers:
  - Each MPI process taking care of a walker

## Work load imbalance

<img src="figs/walkers.svg" alt="walkers.svg" width="90%">
<!--```{image} figs/walkers.svg
:width: 1100px
```-->

- Walkers:
  - Move and Minimization
  - Processing and Communication
- If blocking PtP communication mode would be used:
  - will result in significant loss of resources: <font color='red'>idle time</font> and <font color='red'>synchronization</font>
  - The timelines of walkers would include long idle time of many processes
<div class="callout note">
<strong>Note:</strong>
<span style="color:darkblue">
This is a prime example for which blocking PtP communication is <font color='red'>inappropriate</font>!</span>
</div>

## Nonblocking point-to-point communication

- Call to a nonblocking send/recv procedure <font color='green'>returns straight away</font>. It avoids synchronization so that the following <font color='green'>opportunities</font> can be exploited:
  - Avoiding certain deadlocks
  - Truly bidirectional communication
  - Avoid idle time:
    - Overlapping communication and computation

<img src="figs/nonblocking-send.svg" alt="nonblocking-send.svg" width="90%">
<!--```{image} figs/nonblocking-send.svg
:width: 1100px
```-->

## Standard nonblocking send/receive

```cpp
int MPI_Isend(const void *buf, int count, MPI_Datatype datatype, int dest,
    int tag, MPI_Comm comm, MPI_Request *request)
```

```cpp
int MPI_Irecv(void *buf, int count, MPI_Datatype datatype,
        int source, int tag, MPI_Comm comm, MPI_Request *request)
```


- <font color='green'>request</font>: pointer to variable of type <font color='green'>MPI_Request</font>, will be associated with the corresponding operation
- <font color='red'>Do not reuse sendbuf/recvbuf before MPI_Isend/MPI_Irecv has been completed!!!</font>
  - Return of call does not imply completion
- <font color='green'>MPI_Irecv has no status argument</font>
  - obtained later during completion via <font color='blue'>MPI_Wait*/MPI_Test*</font>

## Nonblocking send and receive variants

- <font color='green'>Completion</font>
  - Return of MPI_I* call does not imply completion
  - Check for completion via <font color='blue'>MPI_Wait*/MPI_Test*</font>
  - Semantics identical to blocking call after successful completion
<table style="border: 1px solid black; text-align: center">
  <tr>
    <th style="text-align: center; background-color: powderblue">nonblocking MPI function</th><th style="text-align: center; background-color: powderblue">blocking MPI function</th><th style="text-align: center; background-color: powderblue">type</th><th style="text-align: center; background-color: powderblue">completes when</th>
  </tr>
    <tr><td style="text-align: left">MPI_Isend </td><td style="text-align: left">MPI_Send </td><td style="text-align: left">synchronous or buffered</td><td style="text-align: left">depends on type       </td></tr>
    <tr><td style="text-align: left">MPI_Ibsend</td><td style="text-align: left">MPI_Bsend</td><td style="text-align: left">buffered               </td><td style="text-align: left">buffer has been copied</td></tr>
    <tr><td style="text-align: left">MPI_Issend</td><td style="text-align: left">MPI_Ssend</td><td style="text-align: left">synchronous            </td><td style="text-align: left">remote starts receive </td></tr>
    <tr><td style="text-align: left">MPI_Irecv </td><td style="text-align: left">MPI_Recv </td><td style="text-align: left">--                     </td><td style="text-align: left">message was received  </td></tr>
</table>

## Test for completion

Two test modes:

- <font color='green'>Blocking</font>
  - `MPI_Wait*`: Wait until the communication has been completed and buffer can safely be reused
- <font color='green'>Nonblocking</font>
  - `MPI_Test*`: Return true (false) if the communication has (not) completed
<div class="callout note">
<strong>Note:</strong>
<span style="color:darkblue">
Despite the naming, the modes both pertain to nonblocking point-to-point communication!</span>
</div>


## Test for completion - single request

- Examining <font color='red'>only one</font> communication handle for completion:
```cpp
int MPI_Wait(MPI_Request *request,
             MPI_Status *status)
```
```cpp
int MPI_Test(MPI_Request *request, int *flag,
             MPI_Status *status)
```
- <font color='green'>request</font>: request handle of type MPI_Request
- <font color='green'>status</font>: status object of type MPI_Status (cf. MPI_Recv)
- <font color='green'>flag</font>:	variable of type int to test for success
<div class="callout note">
<strong>Note:</strong>
<span style="color:darkblue">
Fortran binding of <code>MPI_TEST</code>: <code>flag</code> is a <code>logical</code> variable!</span>
</div>


## Use of MPI_Wait/MPI_Test

- MPI_Wait:
```cpp
MPI_Request request;
MPI_Status status;


MPI_Isend(
  send_buffer, count, MPI_CHAR, 
  dst, 0, MPI_COMM_WORLD, &request);


// do some work… 
// do not use send_buffer

MPI_Wait(&request, &status);

// use send_buffer
```

- MPI_Test:
```cpp
MPI_Request request;
MPI_Status status;
int flag;

MPI_Isend(
  send_buffer, count, MPI_CHAR, 
  dst, 0, MPI_COMM_WORLD, &request);

do {
    // do some work… 
    // do not use send_buffer
    MPI_Test(&request, &flag, &status);
} while (!flag);

// use send_buffer
```

## Wait for completion - all requests in a list

- MPI can handle multiple communication requests
- Wait/Test for completion of multiple requests:
```cpp
int MPI_Waitall(int count, MPI_Request array_of_requests[],
                           MPI_Status *array_of_statuses)
```
```cpp
int MPI_Testall(int count, MPI_Request array_of_requests[],
                int *flag, MPI_Status array_of_statuses[])
```
- Waits for/Tests if all provided requests have been completed

## Use of MPI_Waitall

```cpp
MPI_Request requests[2];
MPI_Status  statuses[2];

MPI_Isend(send_buffer, …, &(requests[0]));
MPI_Irecv(recv_buffer, …, &(requests[1]));

// do some work… 

MPI_Waitall(2, requests, statuses)
// Isend & Irecv have been completed
```
- Arrays of <font color='green'>requests</font> and <font color='green'>statuses</font>
- Two nonblocking calls, so number of elements in the arrays is `2`

## Ghost cell exchange with nonblocking MPI

<div style="text-align: center; font-size: 24px; max-width: 2000px;">
    <span">
        Ghost cell exchange with nonblocking send/recv with all neighbors at once
    </span>
</div> <p></p>

<img src="figs/ghost-cell-5.svg" alt="ghost-cell-5.svg" style="float: right; margin-right: 15px;" width="50%">
<!--```{image} figs/ghost-cell-5.svg
:width: 800px
```-->

Possible implementation:
- <font color='green'>Update cells that need the halo</font>
- <font color='orange'>Copy new data into contiguous send buffers</font>
- <font color='purple'>Start nonblocking receives/sends from/to corresponding neighbors</font>
- <font color='green'>Update local cells that do not need halo cells for boundary conditions ("bulk update")</font>
- <font color='blue'>Wait with MPI_Waitall for all obtained requests to complete</font>
- <font color='orange'>Copy received halo data into ghost cells</font>

<div style="text-align: center; font-size: 24px; max-width: 2000px;">
    <span">
        Opportunity to overlap communication (steps 3-5) with <font color='green'>bulk update</font> (MPI implementation permitting)
    </span>
</div>

## Wait for completion - one or several requests out of a list

- Wait for/Test if <font color='blue'>exactly one</font> request <font color='green'>among many</font> has been completed:
```cpp
int MPI_Waitany(int count, MPI_Request array_of_requests[], int *index,
                           MPI_Status *status)
```
```cpp
int MPI_Testany(int count, MPI_Request array_of_requests[], int *index,
                int *flag, MPI_Status *status)
```
- Wait for/Test if <font color='blue'>at least one</font> request <font color='green'>among many</font> has been completed:
```cpp
int MPI_Waitsome(int incount, MPI_Request array_of_requests[],
                 int *outcount, int array_of_indices[], MPI_Status array_of_statuses[])
```
```cpp
int MPI_Testsome(int incount, MPI_Request array_of_requests[],
                 int *outcount, int array_of_indices[], MPI_Status array_of_statuses[])
```

## Use of MPI_Testany

```cpp
MPI_Request requests[2];
MPI_Status  status;
int finished = 0;

MPI_Isend(send_buffer, …, &(requests[0]));
MPI_Irecv(recv_buffer, …, &(requests[1]));
do {
  // do some work… 
  MPI_Testany(2, requests, &idx, &flag, &status);
  if (flag) { ++finished; }
} while (finished < 2);
```
<div class="callout note">
<strong>Note:</strong>
<span style="color:darkblue">
&#9642; completed requests are automatically set to <code>MPI_REQUEST_NULL</code> <br>
&#9642; completed requests: <code>requests[idx]</code></span>
</div>

## Pitfalls with nonblocking MPI and compiler optimizations

- Fortran:
```fortran
call MPI_IRECV(recvbuf, ..., request, ierror)
call MPI_WAIT(request, status, ierror)
write (*,*) recvbuf
```

<div class="callout note">
<strong>Note:</strong>
<span style="color:darkblue">
MPI might modify recvbuf after <code>MPI_IRECV</code> returns, but the compiler has no idea about this point!</span>
</div>

which may be compiled as
```fortran
call MPI_IRECV(recvbuf, ..., request, ierror)
registerA = recvbuf
call MPI_WAIT(request, status, ierror)
write (*,*) registerA
```
- i.e., old data is written instead of received data!

- Workarounds:
  - `recvbuf` may be allocated in a common block, or
  - calling `MPI_GET_ADDRESS(recvbuf, iaddr_dummy, ierror)` after `MPI_WAIT`
  - <font color='green'>asynchronous</font> attribute



## Array call-by-value in Fortran

- Fortran:
```fortran
call MPI_ISEND(buf(7,:,:), ..., request, ierror)
! other work
call MPI_WAIT(request, status, ierror)
```
<div class="callout note">
<strong>Note:</strong>
<span style="color:darkblue">
&#9642; specified array is non-contiguous <br>
&#9642; compiler generates a temporary array for the function all <br>
&#9642; temp. array is destroyed after <code>MPI_ISEND</code> returns</span>
</div>
- <font color='red'>Do not use non-contiguous sub-arrays in nonblocking calls!</font>
- Use first sub-array element: <code>buf(1,1,9)</code> instead of whole sub-array: <code>buf(:,:,9:13)</code>
<div class="callout note">
<strong>Note:</strong>
<span style="color:darkblue">
data is sent in this time frame, but source array is already lost</span>
</div>
- <font color='red'>Call by reference is absolutely required</font>
  - Call by in-and-out-copy forbidden

## Nonblocking point-to-point communication: summary

- Standard nonblocking send/recv <font color='green'>MPI_Isend()/MPI_Irecv()</font>
  - Return of call does not imply completion of operation
  - Use <font color='green'>MPI_Wait*()/MPI_Test*()</font> to check for completion using <font color='green'>request handles</font>
- All outstanding requests must be completed!
- <font color='green'>Potentials</font>
  - Overlapping of communication with work (not guaranteed by MPI standard)
  - Overlapping send and receive
  - Avoiding synchronization and reducing idle times
- <font color='red'>Caveat</font>: Compiler does not know about asynchronous modification of data

## Quiz:

1. Every nonblocking send or receive <font color='green'>requires</font> a subsequent MPI_Wait* or MPI_Test* call?
   <ol style="list-style-type: lower-alpha;">
   <li>Correct</li>
   <li>Incorrect</li>
   </ol>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> a.
   </details>
2. Can <font color='green'>MPI_Isend</font> be matched with blocking receive (<font color='green'>MPI_Recv</font>)?
   <ol style="list-style-type: lower-alpha;">
   <li>Yes</li>
   <li>No</li>
   </ol>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> a.
   </details>
3. Which one is <font color='green'>not a certain benefit</font> of using nonblocking MPI point-to-point calls?
   <ol style="list-style-type: lower-alpha;">
   <li>Overlapping send and receive</li>
   <li>Avoiding idle times</li>
   <li>Overlapping of communication with work</li>
   </ol>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> c.
   </details>